# KrishiSetu AI - Edge Model Training for PWA (TensorFlow & OpenCV)
This notebook trains a lightweight **MobileNetV2** model to detect crop diseases and exports it to a **TensorFlow.js** (`tfjs`) format so it can run 100% offline in our React webapp right on the farmer's mobile device.

### 📊 Dataset Requirements (For Hackathon):
- **Total Classes:** Minimum 5 classes (e.g., Healthy Paddy, Paddy Leaf Smut, Healthy Cotton, Cotton Aphids, etc.)
- **Images Per Class:** We highly recommend **500 to 1,000 images per class** for a robust mobile model. (Absolute minimum for a demo is 100 images per class).
- **Format:** `.jpg` or `.png` organized into folders named after the class (e.g., `/dataset/paddy_smut/img1.jpg`).

### 🛠️ Image Pre-Processing (OpenCV):
We use OpenCV (`cv2`) in this notebook to dynamically resize, crop, and normalize the images before feeding them into the TensorFlow Tensor. Mobile devices struggle with massive 4K camera photos, so resizing them to `224x224` pixels is critical for fast Edge AI.

### 🚀 Instructions:
1. Upload your dataset to this Colab environment.
2. Update the `dataset_dir` variable in the code below.
3. Run all cells (Shift + Enter).
4. Zip and download the `tfjs_model` folder generated at the end, and place it inside the `public/model/` folder of your React app.


In [ ]:
!pip install tensorflowjs
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import tensorflowjs as tfjs
import os

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# Test OpenCV by loading a sample image (Optional visualization step)
# img = cv2.imread('/content/dataset/sample/img.jpg')
# if img is not None:
#     img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB for TF
#     resized_img = cv2.resize(img_rgb, (224, 224))
#     plt.imshow(resized_img)
#     plt.title("OpenCV Pre-processed Image")
#     plt.show()
print("OpenCV Ready for preprocessing if needed!")


In [ ]:
# Define Dataset Path and Params
dataset_dir = '/content/dataset' # CHANGE THIS to your extracted dataset folder
img_size = (224, 224)
batch_size = 32

print("Loading dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=img_size,
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=img_size,
  batch_size=batch_size)

class_names = train_ds.class_names
print("Classes detected:", class_names)

In [ ]:
# Build the lightweight model
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base model for transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model (Keep epochs low for hackathon speed)
model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
# Export for the Web App!
tfjs_target_dir = '/content/tfjs_model'
tfjs.converters.save_keras_model(model, tfjs_target_dir)

# Save the class names so the app knows what it predicted
with open(os.path.join(tfjs_target_dir, 'classes.json'), 'w') as f:
    import json
    json.dump(class_names, f)

print(f"SUCCESS! Download the {tfjs_target_dir} folder and put it in KrishiSetu-AI/public/model/")